In [1]:
#Tải các thư viện Python dùng cho việc xử lý dữ liệu
import pandas as pd
import numpy as np

print("Đã import thư viện thành công!")

Đã import thư viện thành công!


In [2]:
# Đọc file dữ liệu gốc từ thư mục data
data_path = "../data/RT_IoT2022"  

df = pd.read_csv(data_path)

# Kiểm tra 5 dòng đầu tiên để xem dữ liệu mặt mũi ra sao
print(f"Kích thước ban đầu của dữ liệu: Dòng = {df.shape[0]}, Cột = {df.shape[1]}")
df.head()

Kích thước ban đầu của dữ liệu: Dòng = 123117, Cột = 85


,Unnamed: 0,id.orig_p,id.resp_p,proto,service,flow_duration,fwd_pkts_tot,bwd_pkts_tot,fwd_data_pkts_tot,bwd_data_pkts_tot,...,active.std,idle.min,idle.max,idle.tot,idle.avg,idle.std,fwd_init_window_size,bwd_init_window_size,fwd_last_window_size,Attack_type
0,0,38667,1883,tcp,mqtt,32.011598,9,5,3,3,...,0.0,2.972918e+07,2.972918e+07,2.972918e+07,2.972918e+07,0.0,64240,26847,502,MQTT_Publish
1,1,51143,1883,tcp,mqtt,31.883584,9,5,3,3,...,0.0,2.985528e+07,2.985528e+07,2.985528e+07,2.985528e+07,0.0,64240,26847,502,MQTT_Publish
2,2,44761,1883,tcp,mqtt,32.124053,9,5,3,3,...,0.0,2.984215e+07,2.984215e+07,2.984215e+07,2.984215e+07,0.0,64240,26847,502,MQTT_Publish
3,3,60893,1883,tcp,mqtt,31.961063,9,5,3,3,...,0.0,2.991377e+07,2.991377e+07,2.991377e+07,2.991377e+07,0.0,64240,26847,502,MQTT_Publish
4,4,51087,1883,tcp,mqtt,31.902362,9,5,3,3,...,0.0,2.981470e+07,2.981470e+07,2.981470e+07,2.981470e+07,0.0,64240,26847,502,MQTT_Publish


In [3]:
# Xóa cột 'Unnamed: 0' nếu nó xuất hiện do lúc lưu file bị thừa index cũ
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])
    print("Đã xóa cột 'Unnamed: 0' thừa.")
else:
    print("Không tìm thấy cột 'Unnamed: 0', dữ liệu sạch sẽ.")

Đã xóa cột 'Unnamed: 0' thừa.


In [4]:
# Kiểm tra và xóa các dòng dữ liệu bị trùng lặp hoàn toàn
print(f"Số dòng trước khi xóa trùng: {df.shape[0]}")

df = df.drop_duplicates()

print(f"Số dòng sau khi xóa trùng: {df.shape[0]}")

Số dòng trước khi xóa trùng: 123117
Số dòng sau khi xóa trùng: 117922


In [5]:
# Xóa cột 'bwd_URG_flag_count' vì nó là cột hằng số (chỉ có 1 giá trị duy nhất, không có ích cho phân loại)
constant_column = 'bwd_URG_flag_count'

if constant_column in df.columns:
    df = df.drop(columns=[constant_column])
    print(f"Đã xóa thành công cột hằng số: {constant_column}")
else:
    print(f"Cột {constant_column} không tồn tại trong DataFrame.")

Đã xóa thành công cột hằng số: bwd_URG_flag_count


In [ ]:
#Lọc và loại bỏ các cột có độ tương quan tuyệt đối > 0.95 với nhau 

# Tính ma trận tương quan cho các biến số
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr().abs()

# Lấy phần tam giác trên của ma trận tương quan
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Tìm tên các cột có hệ số tương quan lớn hơn 0.95
to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > 0.95)]

print(f"Các cột có tương quan > 0.95 sẽ bị loại bỏ ({len(to_drop)} cột):")
print(to_drop)

# Thực hiện xóa khỏi DataFrame chính
df = df.drop(columns=to_drop)
print(f"Số cột còn lại sau khi loại bỏ đa cộng tuyến: {df.shape[1]}")

Các cột có tương quan > 0.95 sẽ bị loại bỏ (0 cột):
[]
Số cột còn lại sau khi loại bỏ đa cộng tuyến: 59


In [8]:
#Kiểm tra xem dữ liệu còn ô nào bị trống (NaN/Null) không
missing_count = df.isnull().sum().sum()
print(f"Tổng số giá trị bị thiếu trong toàn bộ dataset: {missing_count}")

# Nếu có giá trị thiếu, ta có thể điền giá trị 0 hoặc xóa đi (ở đây RT-IoT2022 không có missing value)

Tổng số giá trị bị thiếu trong toàn bộ dataset: 0


In [10]:
#Chuyển đổi nhãn dạng chữ (Attack_type) sang dạng số nguyên để thuật toán máy học hiểu được
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

# Kiểm tra tên cột chứa nhãn nhắm tới (thường là 'Attack_type' hoặc tương tự)
target_column = 'Attack_type' 

if target_column in df.columns:
    df[target_column] = encoder.fit_transform(df[target_column])
    print(f"Đã mã hóa thành công cột mục tiêu '{target_column}' sang dạng số.")
    
    # In ra vài ánh xạ nhãn để dễ theo dõi
    class_mapping = dict(zip(encoder.classes_, range(len(encoder.classes_))))
    print("Ví dụ một vài nhãn sau khi mã hóa:", list(class_mapping.items())[:5])
else:
    print(f"Không tìm thấy cột mục tiêu {target_column}, hãy kiểm tra lại tên cột!")

Đã mã hóa thành công cột mục tiêu 'Attack_type' sang dạng số.
Ví dụ một vài nhãn sau khi mã hóa: [('ARP_poisioning', 0), ('DDOS_Slowloris', 1), ('DOS_SYN_Hping', 2), ('MQTT_Publish', 3), ('Metasploit_Brute_Force_SSH', 4)]


In [14]:
#Mã hóa các đặc trưng dạng chữ còn sót lại và Chuẩn hóa (Scale) toàn bộ dữ liệu
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pandas as pd

# 1. Tách X (đặc trưng) và y (mục tiêu)
# Dùng .copy() để tạo bản sao độc lập, tránh ảnh hưởng đến df gốc
X = df.drop(columns=[target_column]).copy()
y = df[target_column].copy()

# 2. Xử lý các cột còn chứa chữ (như 'proto', 'service',...) trong X
cat_cols = X.select_dtypes(include=['object', 'category']).columns
print(f"Các cột dạng chữ phát hiện được và đang tiến hành mã hóa: {list(cat_cols)}")

le_X = LabelEncoder()
for col in cat_cols:
    X[col] = le_X.fit_transform(X[col].astype(str))

# 3. Khởi tạo bộ chuẩn hóa
scaler = StandardScaler()

# 4. Thực hiện chuẩn hóa (lúc này X chắc chắn 100% là số nên sẽ không báo lỗi 'tcp' nữa)
X_scaled = scaler.fit_transform(X)

# 5. Chuyển X sau khi chuẩn hóa về lại dạng DataFrame để dễ nhìn
X_df = pd.DataFrame(X_scaled, columns=X.columns)

# 6. Ghép X_df (đặc trưng đã xử lý) và y (nhãn đã xử lý ở cell 8) lại với nhau
df_processed = pd.concat([X_df, y.reset_index(drop=True)], axis=1)

print("\nĐÃ XỬ LÝ THÀNH CÔNG! Dữ liệu đã sẵn sàng.")
df_processed.head()

C:\Users\w10\AppData\Local\Temp\ipykernel_23884\2522947518.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=['object', 'category']).columns


Các cột dạng chữ phát hiện được và đang tiến hành mã hóa: ['proto', 'service']

ĐÃ XỬ LÝ THÀNH CÔNG! Dữ liệu đã sẵn sàng.


,id.orig_p,id.resp_p,proto,service,flow_duration,fwd_pkts_tot,bwd_pkts_tot,fwd_data_pkts_tot,fwd_pkts_per_sec,down_up_ratio,...,bwd_bulk_rate,active.min,active.max,active.tot,active.avg,idle.std,fwd_init_window_size,bwd_init_window_size,fwd_last_window_size,Attack_type
0,0.196030,0.155128,-0.339614,2.552933,0.221961,0.303454,0.090645,0.079306,-0.947016,-0.889348,...,-0.0722,2.015724,0.683097,0.167055,1.293431,-0.042378,3.069373,2.363506,-0.038155,3
1,0.853840,0.155128,-0.339614,2.552933,0.220953,0.303454,0.090645,0.079306,-0.947016,-0.889348,...,-0.0722,1.776926,0.600403,0.145904,1.139110,-0.042378,3.069373,2.363506,-0.038155,3
2,0.517343,0.155128,-0.339614,2.552933,0.222846,0.303454,0.090645,0.079306,-0.947016,-0.889348,...,-0.0722,2.015244,0.682930,0.167013,1.293121,-0.042378,3.069373,2.363506,-0.038155,3
3,1.367919,0.155128,-0.339614,2.552933,0.221563,0.303454,0.090645,0.079306,-0.947016,-0.889348,...,-0.0722,1.794764,0.606580,0.147484,1.150637,-0.042378,3.069373,2.363506,-0.038155,3
4,0.850888,0.155128,-0.339614,2.552933,0.221101,0.303454,0.090645,0.079306,-0.947016,-0.889348,...,-0.0722,1.832701,0.619717,0.150844,1.175153,-0.042378,3.069373,2.363506,-0.038155,3


In [16]:
# Lưu dataframe đã sạch sẽ và xử lý xong vào thư mục data/processed/ để đem đi huấn luyện mô hình
import os

# Tạo thư mục processed nếu chưa có
os.makedirs("../data/processed", exist_ok=True)

output_file = "../data/processed/RT_IoT2022_processed.csv"
df_processed.to_csv(output_file, index=False)

print(f"Đã lưu thành công file dữ liệu sạch tại: {output_file}")
print(f"Kích thước dữ liệu cuối cùng: Dòng = {df_processed.shape[0]}, Cột = {df_processed.shape[1]}")

Đã lưu thành công file dữ liệu sạch tại: ../data/processed/RT_IoT2022_processed.csv
Kích thước dữ liệu cuối cùng: Dòng = 117922, Cột = 59
